# 🧠 Latih Ulang Model diBISAlitas (YOLOv8) — Colab siap-pakai

Notebook ini melatih model deteksi isyarat (BISINDO / Hijaiyah) dan meng-export ke:
- **ONNX** → untuk web (`web/public/models/...`)
- **TFLite (float32)** → untuk mobile (`mobile/assets/Machine Learning/...`)

Cara pakai: **Runtime → Change runtime type → GPU (T4)**, isi sel *Parameter*, lalu **Runtime → Run all**.

> Penting soal urutan label ada di sel setelah download dataset — WAJIB dibaca.

## 1. Cek GPU

In [ ]:
!nvidia-smi

## 2. Install dependensi

In [ ]:
!pip install -q ultralytics roboflow

## 3. Parameter — ISI DI SINI

Ambil `ROBOFLOW_*` dari halaman dataset Roboflow: tombol **Download Dataset → format YOLOv8 → show download code**. Di situ tertera workspace, project, dan version.

- `MODEL_NAME`: `"bisindo"` atau `"hijaiyah"` (menentukan nama file output).
- Naikkan `EPOCHS` (mis. 150) bila datanya banyak agar akurasi lebih tinggi.

In [ ]:
ROBOFLOW_API_KEY = "PASTE_API_KEY_KAMU"
ROBOFLOW_WORKSPACE = "nama-workspace"
ROBOFLOW_PROJECT   = "nama-project"
ROBOFLOW_VERSION   = 1

MODEL_NAME = "bisindo"   # atau "hijaiyah"
EPOCHS = 120
IMGSZ  = 640
BATCH  = 16

## 4. Download dataset dari Roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov8")
DATA_YAML = dataset.location + "/data.yaml"
print("data.yaml:", DATA_YAML)

## 5. ⚠️ Cek URUTAN LABEL (SANGAT PENTING)

Aplikasi memetakan hasil model ke label lewat **indeks kelas**. Urutan `names` di bawah **harus sama** dengan konstanta label di kode:
- Web BISINDO: `web/src/constants/bisindoLabels.ts` (`BISINDO_LABELS`)
- Mobile BISINDO: `mobile/lib/core/constants/bisindo_labels.dart`
- (Hijaiyah: `signLabels.ts` `HIJAIYAH_LABELS` & `sign_labels.dart`)

Kalau urutannya beda, **samakan konstanta di kode** dengan daftar ini (indeks 0..n).

In [ ]:
import yaml
with open(DATA_YAML) as f:
    d = yaml.safe_load(f)
names = d["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=int)]
print("Jumlah kelas:", len(names))
for i, n in enumerate(names):
    print(i, n)

## 6. Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano — ringan untuk web & HP
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=30,
    name=MODEL_NAME,
    seed=0,
)

## 7. Evaluasi (mAP)

In [ ]:
best = f"runs/detect/{MODEL_NAME}/weights/best.pt"
m = YOLO(best)
metrics = m.val()
print("mAP50-95:", round(float(metrics.box.map), 4))
print("mAP50   :", round(float(metrics.box.map50), 4))

## 8. Export ONNX (web) + TFLite (mobile)

In [ ]:
# ONNX untuk web (onnxruntime-web, NCHW)
m.export(format="onnx", opset=12, imgsz=IMGSZ, simplify=True)

# TFLite float32 untuk mobile (tflite_flutter, NHWC)
m.export(format="tflite", imgsz=IMGSZ)

## 9. Kumpulkan & unduh file model

In [ ]:
import glob, os, shutil

wdir = f"runs/detect/{MODEL_NAME}/weights"
onnx_files = glob.glob(f"{wdir}/*.onnx")
tflite_files = glob.glob(f"{wdir}/**/*float32.tflite", recursive=True) or glob.glob(f"{wdir}/**/*.tflite", recursive=True)
print("ONNX  :", onnx_files)
print("TFLite:", tflite_files)

os.makedirs("/content/hasil_model", exist_ok=True)
if onnx_files:
    shutil.copy(onnx_files[0], "/content/hasil_model/model.onnx")
if tflite_files:
    shutil.copy(tflite_files[0], f"/content/hasil_model/model_{MODEL_NAME}_detect.tflite")

print("\nIsi folder hasil:")
for p in glob.glob("/content/hasil_model/*"):
    print(" -", p)

In [ ]:
from google.colab import files
for p in glob.glob("/content/hasil_model/*"):
    files.download(p)

## 10. Pasang ke proyek (drop-in)

**BISINDO:**
- `model.onnx` → timpa `web/public/models/Bisindo/model.onnx`
- `model_bisindo_detect.tflite` → timpa `mobile/assets/Machine Learning/model_bisindo_detect.tflite`

**Hijaiyah:**
- `model.onnx` → timpa `web/public/models/Hijayah/model.onnx`
- `model_hijaiyah_detect.tflite` → timpa `mobile/assets/Machine Learning/model_hijaiyah_detect.tflite`

Lalu **cek urutan label** (sel #5): kalau berbeda dari konstanta di kode, samakan `BISINDO_LABELS` / `bisindo_labels.dart` (atau Hijaiyah) agar indeks kelas cocok.

Tips akurasi: dataset **≥ ratusan gambar per kelas & seimbang**, EPOCHS 120-200, dan pencahayaan/latar bervariasi. Data yang cukup jauh lebih menentukan daripada jumlah epoch.